# Transformer Chatbot from Scratch

This notebook implements a complete transformer architecture for conversational AI. We'll build every component from scratch to understand how attention mechanisms, positional encoding, and sequence-to-sequence modeling work.

## Architecture
- **Encoder**: Processes the input message
- **Decoder**: Generates the response token by token
- **Attention**: The core mechanism that allows the model to focus on relevant parts of the input

Let's start by setting up our environment!

In [1]:
# Install dependencies (run once)
# !pip install torch transformers datasets tqdm

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer
from datasets import load_dataset
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check for Apple Silicon GPU (MPS), CUDA, or fallback to CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon GPU (MPS)")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("Using NVIDIA GPU (CUDA)")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"PyTorch version: {torch.__version__}")

# Set random seed for reproducibility
torch.manual_seed(42)

# Model save path (local)
MODEL_PATH = 'best_chatbot_model.pt'

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


/opt/anaconda3/envs/transformer/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Apple Silicon GPU (MPS)
PyTorch version: 2.2.2


In [2]:
# Verify MPS is working (Apple Silicon GPU)
if device.type == 'mps':
    # Quick test to ensure MPS is functioning
    test_tensor = torch.randn(2, 2, device=device)
    print(f"MPS test successful! Tensor device: {test_tensor.device}")
    
print(f"Model will be saved to: {MODEL_PATH}")

MPS test successful! Tensor device: mps:0
Model will be saved to: best_chatbot_model.pt


In [ ]:
# Load the DailyDialog dataset (using community-maintained version)
print("Loading DailyDialog dataset...")
dataset = load_dataset("roskoN/dailydialog")

print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")

# Let's look at an example dialogue
print("\nExample dialogue:")
for i, utterance in enumerate(dataset['train'][0]['dialog'][:4]):
    speaker = "A" if i % 2 == 0 else "B"
    print(f"  {speaker}: {utterance}")

Loading DailyDialog dataset...


RuntimeError: Dataset scripts are no longer supported, but found dailydialog.py

## Summary

Congratulations! You've built a complete transformer chatbot from scratch! Here's what we covered:

1. **Data Preparation**: Loaded DailyDialog dataset and created input-response pairs
2. **Positional Encoding**: Sinusoidal position embeddings to inject sequence order
3. **Multi-Head Attention**: Scaled dot-product attention with multiple heads
4. **Encoder**: Self-attention layers to encode the input message
5. **Decoder**: Masked self-attention + cross-attention to generate responses
6. **Training**: Cross-entropy loss with teacher forcing
7. **Inference**: Greedy and temperature-based decoding

### Tips for Better Results

- **Train longer**: Increase `EPOCHS` to 10-20 for better quality
- **Use GPU**: Training is much faster on CUDA
- **Increase model size**: Larger `D_MODEL`, `N_LAYERS`, `D_FF` improve capacity
- **More data**: Try combining multiple conversation datasets
- **Fine-tune**: Start from a pre-trained model for better results

### Key Concepts to Remember

- **Self-Attention**: Allows each token to attend to all other tokens
- **Cross-Attention**: Decoder attends to encoder output
- **Causal Masking**: Prevents decoder from seeing future tokens
- **Positional Encoding**: Injects sequence order information
- **Layer Normalization**: Stabilizes training